# Сегментация

**AI2 · Computer Vision** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~15 min

## Введение

Ранее мы изучали обнаружение объектов, которое позволяет находить объекты на изображении, предсказывая их *ограничивающие рамки*. Однако для некоторых задач нам нужны не только рамки, но и более точная локализация объектов. Эта задача называется **сегментацией**.


Сегментацию можно рассматривать как **классификацию пикселей**, где для **каждого** пикселя изображения мы должны предсказать его класс (*фон* является одним из классов). Существует два основных алгоритма сегментации:

* **Семантическая сегментация** определяет только класс пикселя, не различая отдельные объекты одного и того же класса.
* **Сегментация экземпляров** разделяет классы на отдельные экземпляры.

Для сегментации экземпляров эти овцы считаются разными объектами, но для семантической сегментации все овцы представлены одним классом.

![](ms-assets/ai/instance_vs_semantic.eee9812bebf8cd45.webp)

> Изображение из [этой статьи](https://nirmalamurali.medium.com/image-classification-vs-semantic-segmentation-vs-instance-segmentation-625c33a08d50)

Существуют различные нейронные архитектуры для сегментации, но все они имеют одинаковую структуру. В некотором смысле, она похожа на автоэнкодер, о котором вы узнали ранее, но вместо восстановления исходного изображения наша цель — восстановить **маску**. Таким образом, сеть для сегментации состоит из следующих частей:

* **Энкодер** извлекает признаки из входного изображения.
* **Декодер** преобразует эти признаки в **маску изображения**, с тем же размером и количеством каналов, соответствующих количеству классов.

![](ms-assets/ai/segm.92442f2cb42ff4fa.webp)

> Изображение из [этой публикации](https://arxiv.org/pdf/2001.05566.pdf)

Особое внимание следует уделить функции потерь, используемой для сегментации. При использовании классических автоэнкодеров нам нужно измерить сходство между двумя изображениями, и для этого можно использовать среднеквадратичную ошибку (MSE). В сегментации каждый пиксель целевой маски изображения представляет номер класса (закодированный в формате one-hot вдоль третьего измерения), поэтому необходимо использовать функции потерь, специфичные для классификации, такие как кросс-энтропия, усреднённая по всем пикселям. Если маска бинарная, используется **бинарная кросс-энтропия** (BCE).

> ✅ One-hot кодирование — это способ преобразования метки класса в вектор длиной, равной количеству классов. Ознакомьтесь с [этой статьёй](https://datagy.io/sklearn-one-hot-encode/) о данной технике.



## Сегментация в медицинской визуализации

В этом уроке мы увидим сегментацию в действии, обучая сеть распознавать невусы (родинки) на медицинских изображениях. Мы будем использовать [базу данных PH<sup>2</sup>](https://www.fc.up.pt/addi/ph2%20database.html) дерматоскопических изображений в качестве источника данных. Этот набор данных содержит 200 изображений трёх классов: типичный невус, атипичный невус и меланома. Все изображения также содержат соответствующую **маску**, которая выделяет невус.

> ✅ Этот метод особенно подходит для такого типа медицинской визуализации, но какие ещё реальные приложения вы могли бы представить?

![navi](ms-assets/ai/navi.2f20b727910110ea.webp)

> Изображение из базы данных PH<sup>2</sup>

Мы обучим модель сегментировать любой невус от его фона.



## Упражнения: Семантическая сегментация

Откройте ноутбуки ниже, чтобы узнать больше о различных архитектурах семантической сегментации, попрактиковаться в работе с ними и увидеть их в действии.

* [Semantic Segmentation Pytorch](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/12-Segmentation/SemanticSegmentationPytorch.ipynb)
* [Semantic Segmentation TensorFlow](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/12-Segmentation/SemanticSegmentationTF.ipynb)



## Заключение

Сегментация — это очень мощная техника классификации изображений, которая выходит за рамки ограничивающих рамок и позволяет классифицировать пиксели. Она используется в медицинской визуализации и других приложениях.



## Задание

Сегментация тела — это лишь одна из распространённых задач, которые можно выполнять с изображениями людей. Другие важные задачи включают **обнаружение скелета** и **определение позы**. Попробуйте библиотеку [OpenPose](https://github.com/CMU-Perceptual-Computing-Lab/openpose), чтобы увидеть, как можно использовать определение позы.



## Обзор и самостоятельное изучение

Эта [статья в Википедии](https://wikipedia.org/wiki/Image_segmentation) предлагает хороший обзор различных приложений данной техники. Узнайте больше самостоятельно о поддомейнах сегментации экземпляров и паноптической сегментации в этой области исследований.



## Задание

В этой лабораторной работе попробуйте **сегментацию человеческого тела** с использованием [Segmentation Full Body MADS Dataset](https://www.kaggle.com/datasets/tapakah68/segmentation-full-body-mads-dataset) из Kaggle.



---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Mask to bounding box

●○○ · ~8 min

> A segmentation mask implies a detection box: the tight box around the foreground pixels is [min col, min row, max col + 1, max row + 1] in xyxy convention — the '+1' makes x2, y2 exclusive so the box actually covers the extreme pixel. np.nonzero hands you the coordinates directly. Decide the empty-mask convention up front; here it is None.

Implement `mask_to_bbox(mask)` for a binary (H, W) mask of 0/1 → `[x1, y1, x2, y2]` (list of ints) tightly enclosing all 1-pixels: x1 = leftmost column, y1 = topmost row, x2 = one past the rightmost column, y2 = one past the bottom row. If the mask has no foreground pixels, return `None`.

<details><summary>Hint</summary>

ys, xs = np.nonzero(mask); if xs.size == 0 → None; else [xs.min(), ys.min(), xs.max() + 1, ys.max() + 1] as ints.

</details>


In [ ]:
import numpy as np

def mask_to_bbox(mask):
    m = np.asarray(mask)
    # your code
    return [0, 0, 0, 0]

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    h, w = int(rng.integers(3, 13)), int(rng.integers(3, 13))\n    p = float(rng.choice([0.0, 0.2, 0.4, 0.7]))\n    return {\"mask\": (rng.random((h, w)) < p).astype(int)}\n\ndef mask_to_bbox(mask):\n    m = np.asarray(mask)\n    ys, xs = np.nonzero(m)\n    if xs.size == 0:\n        return None\n    return [int(xs.min()), int(ys.min()),\n            int(xs.max()) + 1, int(ys.max()) + 1]", "mask_to_bbox", 40)

### Task 2 — Pixel accuracy + Dice

●●○ · ~10 min

> Two metrics dominate binary segmentation: pixel accuracy — the fraction of correctly classified pixels, which is misleading under class imbalance — and Dice = 2·|P∩G| / (|P| + |G|), effectively the F1 score of masks. The edge case everyone forgets: two empty masks are a perfect match, so Dice = 1.

Implement `seg_metrics(pred, gt)` for binary (H, W) masks of 0/1 → `{'pixel_acc': float, 'dice': float}`. Dice = 2·|P∩G| / (|P| + |G|); if both masks are empty, dice = 1.0.

<details><summary>Hint</summary>

np.logical_and(pred, gt).sum() gives the intersection; compute the denominator first and branch on it being 0.

</details>


In [ ]:
import numpy as np

def seg_metrics(pred, gt):
    p = np.asarray(pred)
    g = np.asarray(gt)
    # your code
    return {"pixel_acc": 0.0, "dice": 0.0}

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    h, w = int(rng.integers(3, 11)), int(rng.integers(3, 11))\n    p = float(rng.uniform(0.15, 0.6))\n    pred = (rng.random((h, w)) < p).astype(int)\n    gt = (rng.random((h, w)) < p).astype(int)\n    # force empty-mask edge cases ~30% of the time — the\n    # both-empty → dice=1 rule must actually be exercised\n    u = rng.random()\n    if u < 0.10:\n        pred[:] = 0; gt[:] = 0\n    elif u < 0.20:\n        gt[:] = 0\n    elif u < 0.30:\n        pred[:] = 0\n    return {\"pred\": pred, \"gt\": gt}\n\ndef seg_metrics(pred, gt):\n    p = np.asarray(pred)\n    g = np.asarray(gt)\n    acc = float((p == g).mean())\n    inter = float(np.logical_and(p, g).sum())\n    s = float(p.sum() + g.sum())\n    dice = 1.0 if s == 0.0 else 2.0 * inter / s\n    return {\"pixel_acc\": acc, \"dice\": dice}", "seg_metrics", 40)

### Task 3 — Count blobs (connected components)

●●● · ~45 min

> Counting objects in a binary mask is the connected-components problem: every unvisited foreground cell starts a new blob, then a flood fill marks everything connected to it. Flood fill: push the cell onto a stack; while the stack is non-empty pop a cell, and for each of its 4 neighbors that is in-bounds, a `1`, and unvisited — mark it visited and push it. With 4-connectivity only up/down/left/right neighbors join a blob — diagonal contact does not count. Plain Python loops are plenty for tiny grids; don't reach for scipy.

Implement `count_blobs(grid)`: number of connected components of 1-cells in a binary (H, W) grid under 4-connectivity (diagonals do NOT connect). Return a plain int.

<details><summary>Hint</summary>

Keep a `seen` mask; scan cells, and when you hit an unvisited 1, increment the counter and BFS/DFS from it, marking visited neighbors.

</details>


In [ ]:
import numpy as np

def count_blobs(grid):
    g = np.asarray(grid)
    # your code — flood-fill / union-find connected components
    return int(g.sum())  # wrong — counts cells, not blobs

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    h, w = int(rng.integers(3, 11)), int(rng.integers(3, 11))\n    p = float(rng.choice([0.2, 0.35, 0.5, 0.65]))\n    return {\"grid\": (rng.random((h, w)) < p).astype(int)}\n\ndef count_blobs(grid):\n    g = np.asarray(grid)\n    h, w = g.shape\n    seen = np.zeros((h, w), dtype=bool)\n    cnt = 0\n    for i in range(h):\n        for j in range(w):\n            if g[i, j] and not seen[i, j]:\n                cnt += 1\n                seen[i, j] = True\n                stack = [(i, j)]\n                while stack:\n                    x, y = stack.pop()\n                    for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):\n                        nx, ny = x + dx, y + dy\n                        if (0 <= nx < h and 0 <= ny < w\n                                and g[nx, ny] and not seen[nx, ny]):\n                            seen[nx, ny] = True\n                            stack.append((nx, ny))\n    return cnt", "count_blobs", 40)

---

**Go deeper**
- [Исходный урок (Microsoft, MIT)](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/12-Segmentation)
- [Учебник ML (ШАД): свёрточные нейросети](https://contest.yandex.ru/tracks/ml/deep-learning-architectures/convolutional-neural-networks)